In [ ]:
#importar as bibliotecas
import requests
from bs4 import BeautifulSoup
import pandas as pd

#google sheets
!pip install gspread gspread_dataframe
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default

#autenticacao google
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

#incluir o link de onde estao os dados
url = "https://www.fundamentus.com.br/fii_resultado.php"

#simula um navegador comum
headers = {
    "User-Agent": "Mozilla/5.0"
}

#baixa o conteúdo da página
pagina = requests.get(url, headers=headers)

#organiza o HTML para facilitar a leitura
soup = BeautifulSoup(pagina.text, "html.parser")

#lacaliza a tabela dos FIIs
tabela = soup.find("table", id="tabelaResultado")

#pega todas as linhas da tabela
linhas = tabela.find_all("tr")

#lista para armazenar os dados
dados = []

#ignora a primeira linha (cabeçalho)
for linha in linhas[1:]:

    colunas = linha.find_all("td") #divide a linha em colunas individuais

    ticker = colunas[0].text
    segmento = colunas[1].text
    preco = float(colunas[2].text.replace('.','').replace(',','.'))
    dy = colunas[4].text
    p_vp = colunas[5].text

    #converte P/VP para número
    try:
        p_vp_numero = float(p_vp.replace(",", "."))
    except ValueError:
        p_vp_numero = 1.0

    #classificar o fundo
    if p_vp_numero > 1:
        status = "Alta"
    elif p_vp_numero < 1:
        status = "Baixa"
    else:
        status = "Neutro"

    #adicionaa linha na lista que foi criada acima e que estava vazia ( dados[])
    dados.append([
        ticker,
        preco,
        dy,
        p_vp,
        status,
        segmento
    ])

#criar DataFrame
df_fiis = pd.DataFrame(
    dados,
    columns=[
        "Fundo",
        "Preço",
        "DY",
        "P/VP",
        "Status",
        "Segmento"
    ]
)

planilha = gc.create("FIIs Project")
aba = planilha.sheet1
set_with_dataframe(aba, df_fiis)

print("Dados enviados com sucesso para o Google Sheets!")

Dados enviados com sucesso para o Google Sheets!
